<a href="https://colab.research.google.com/github/braim/nids-tta/blob/main/colab/3-ccta-kan-ae/NIDS_3_1_0_0_CCTA_KAN_AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.1.0 MinMax and Kan not ranged

In [ ]:
!pip install git+https://github.com/Blealtan/efficient-kan.git

  Cloning https://github.com/Blealtan/efficient-kan.git to /tmp/pip-req-build-ytdihrpd
  Running command git clone --filter=blob:none --quiet https://github.com/Blealtan/efficient-kan.git /tmp/pip-req-build-ytdihrpd
  Resolved https://github.com/Blealtan/efficient-kan.git to commit 7b6ce1c87f18c8bc90c208f6b494042344216b11
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# Install necessary libraries and import
import sys
import subprocess
import importlib.util

# --- Auto-Install Dependencies ---
packages = {
    'efficient_kan': "git+https://github.com/Blealtan/efficient-kan.git",
    'polars': "polars",
    'kagglehub': "kagglehub"
}

for lib, cmd in packages.items():
    if importlib.util.find_spec(lib) is None:
        print(f"Installing {lib}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", cmd])

import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve
from efficient_kan import KAN
import kagglehub
import os
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
import random



In [ ]:
@dataclass
class Hyperparameters:
    # ── Reproducibility ───────────────────────────────────────────────────────────
    seed: int = 42
    sample_n: int = 100000
    latent_dim: int = 8
    batch_size: int = 128
    memory_capacity: int = 2000
    pretrain_epochs: int = 15
    adapt_epochs: int = 1
    learning_rate: float = 1e-3

config = Hyperparameters()
print(f"Hyperparameters: {config}")

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
np.random.seed(config.seed)
random.seed(config.seed)
torch.backends.cudnn.deterministic = True

In [ ]:

# ==========================================
# 1. POLARS DATA LOADING & PREPROCESSING
# ==========================================
def load_nf_dataset_polars(dataset_name, sample_n=config.sample_n, scaler=None):
    """
    Loads NF-v3 dataset using Polars (Fast!), cleans it, and scales it.
    """
    print(f"⚡ Loading {dataset_name} with Polars...")

    # 1. Define columns to DROP (Metadata & Labels)
    # These exist in standard NF-v3 datasets
    drop_cols = [
        'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT',
        'Attack', 'Label', 'Dataset', 'Date', 'attack', 'label'
    ]

    try:
        # Download
        download_path = kagglehub.dataset_download(dataset_name)
        csv_files = [os.path.join(r, d, files) for r, d, files in os.walk(download_path) for f in files if f.endswith('.csv')]
        # The line above was slightly buggy in the original ('os.path.join(r, d, files)' should be 'os.path.join(r, f)'), let's fix it:
        csv_files = [os.path.join(r, f) for r, d, files in os.walk(download_path) for f in files if f.endswith('.csv')]
        if not csv_files: raise FileNotFoundError("No CSV found")
        filepath = csv_files[0]

        # 2. Lazy Scan (doesn't load into RAM yet)
        q = pl.scan_csv(filepath)

        # 3. Filter Columns that actually exist
        columns = q.collect_schema().names()
        actual_drop = [c for c in drop_cols if c in columns]

        # 4. Select Features & Cast to Float32
        # We drop the metadata, but we need to keep 'Label' separately for y
        q_features = q.drop(actual_drop)

        # 5. Collect & Sample (Load into RAM)
        # If dataset is huge, we fetch a random sample immediately
        df = q.collect()

        if sample_n and df.height > sample_n:
            df = df.sample(n=sample_n, seed=42)

        # 6. Separate X and y
        y = df['Label'].to_numpy()
        X_df = df.drop([c for c in df.columns if c in drop_cols])

        # --- FIX: Handle Infinity / NaN before casting ---
        X = X_df.to_numpy()
        X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)
        X = X.astype(np.float32)

        # 7. Scaling (Critical for Autoencoders)
        if scaler is None:
            scaler = MinMaxScaler()
            X = scaler.fit_transform(X)
        else:
            X = scaler.transform(X) # Use source scaler for targets!

        print(f"   -> Shape: {X.shape} | Attack Rate: {np.mean(y):.2%}")
        return X, y, scaler

    except Exception as e:
        print(f"   Error loading {dataset_name}: {e}")


# ==========================================
# MODEL: KAN-AUTOENCODER
# ==========================================
class KanAE(nn.Module):
    def __init__(self, input_dim=43, latent_dim=config.latent_dim):
        super(KanAE, self).__init__()
        self.encoder = KAN([input_dim, 32, 16, latent_dim])
        self.decoder = KAN([latent_dim, 16, 32, input_dim])

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

# ==========================================
# 3. REPLAY BUFFER (For Continual Learning)
# ==========================================
class ReplayBuffer:
    def __init__(self, capacity=config.memory_capacity):
        self.buffer = []
        self.capacity = capacity

    def add(self, batch_tensor):
        # Reservoir Sampling
        for i in range(batch_tensor.shape[0]):
            if len(self.buffer) < self.capacity:
                self.buffer.append(batch_tensor[i].detach().cpu())
            else:
                idx = np.random.randint(0, self.capacity)
                self.buffer[idx] = batch_tensor[i].detach().cpu()

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            return torch.stack(self.buffer)
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        return torch.stack([self.buffer[i] for i in indices])

# ==========================================
# 4. TRAINING & EVALUATION UTILS
# ==========================================
def evaluate_anomaly_detection(model, loader, device='cuda', desc="Test"):
    model.eval()
    mse_loss = nn.MSELoss(reduction='none')
    all_errors = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            x_recon, _ = model(x)
            loss = torch.mean(mse_loss(x_recon, x), dim=1)
            all_errors.extend(loss.cpu().numpy())
            all_labels.extend(y.numpy())

    errors = np.array(all_errors)
    labels = np.array(all_labels)
    if np.isnan(errors).any(): errors = np.nan_to_num(errors)

    precisions, recalls, thresholds = precision_recall_curve(labels, errors)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1] if len(thresholds) > 0 else 0.0

    preds = (errors >= best_thresh).astype(int)
    best_acc = accuracy_score(labels, preds)

    print(f"   [{desc}] Best F1: {best_f1:.4f} | Acc: {best_acc:.2%} | Thresh: {best_thresh:.4f}")
    return best_acc, best_f1

def train_continual(model, loader, buffer, epochs=1, device='cuda', mode='source'):
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
    criterion = nn.MSELoss()
    model.train()
    model.to(device)

    mse_none = nn.MSELoss(reduction='none')

    for epoch in range(epochs):
        total_loss = 0
        batch_count = 0

        # Unpack x, y to allow online evaluation
        for x, y in loader:
            x = x.to(device)

            if mode == 'adapt':
                # --- Step 1: Input & Online Inference (The Assessment) ---
                model.eval()
                with torch.no_grad():
                    x_recon, _ = model(x)
                    # Reconstruction Error (MSE per sample)
                    errors = torch.mean(mse_none(x_recon, x), dim=1)

                    # Metric Logging: Online F1 Score (Using Ground Truth y)
                    if y is not None:
                        y_np = y.cpu().numpy()
                        errors_np = errors.cpu().numpy()
                        # Oracle check: What is the best possible F1 for this batch?
                        prec, rec, _ = precision_recall_curve(y_np, errors_np)
                        f1s = 2 * (prec * rec) / (prec + rec + 1e-10)
                        best_f1 = np.max(f1s) if len(f1s) > 0 else 0.0
                        # Print sparingly (e.g., first batch or accumulate) - Printing every batch for log
                        print(f"       [Online Adapt] Batch F1: {best_f1:.4f}")

                model.train()

                # --- Step 2: Self-Supervised Filtering (The \"Safety Valve\") ---
                # Logic: Dynamic Threshold (Median of current batch errors)
                threshold = torch.median(errors)

                # Selection: Keep Pseudo-Benign (error < threshold)
                mask_benign = errors < threshold
                x_pseudo_benign = x[mask_benign]

                if x_pseudo_benign.size(0) == 0:
                    continue

                # --- Step 3: Memory Anchoring (The \"Anti-Forgetting\" Mix) ---
                # Retrieval: Pull random batch from Source Buffer
                x_train = x_pseudo_benign
                if len(buffer.buffer) > 0:
                    # Match size of pseudo-benign to balance the update
                    n_replay = min(x_pseudo_benign.size(0), len(buffer.buffer))
                    x_replay = buffer.sample(batch_size=n_replay).to(device)
                    # Combination
                    x_train = torch.cat([x_pseudo_benign, x_replay], dim=0)

                # --- Step 4: Gradient Adaptation & Buffer Expansion (The Update) ---
                # Optimization
                optimizer.zero_grad()
                x_recon_train, _ = model(x_train)
                loss = criterion(x_recon_train, x_train)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                batch_count += 1

                # Memory Update: Add small random fraction of Pseudo-Benign to Buffer
                # We take ~5% of the most confident (lowest error) samples to avoid buffer flooding
                n_add = max(1, int(0.05 * x_pseudo_benign.size(0)))
                # Sort by error (already calculated) to find 'most confident'
                # We need errors for pseudo_benign only
                errors_benign = errors[mask_benign]
                _, sorted_idx = torch.sort(errors_benign, descending=False)
                top_indices = sorted_idx[:n_add]

                # Add to buffer
                buffer.add(x_pseudo_benign[top_indices].detach().cpu())

            else:
                # --- Source Training Mode ---
                optimizer.zero_grad()
                x_recon, _ = model(x)
                loss = criterion(x_recon, x)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                batch_count += 1

                # Add to buffer (Store Source data)
                if mode == 'source':
                    buffer.add(x.detach().cpu())

        avg_loss = total_loss / max(1, batch_count)
        if mode == 'source':
            print(f"   Epoch {epoch+1} Loss: {avg_loss:.4f}")
        else:
            print(f"   Adaptation Epoch {epoch+1} Complete. Avg Update Loss: {avg_loss:.4f}")


Installing efficient_kan...


In [ ]:

# ==========================================
# 5. INITIALIZATION & DATA LOADING
# ==========================================
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using {device}. Starting KAN-IDS Experiment...\n")

# 1. LOAD DATASETS
X_src, y_src, scaler = load_nf_dataset_polars('seyhed/nf-unsw-nb15-v3')
X_tgt1, y_tgt1, _    = load_nf_dataset_polars('seyhed/nf-ton-iot-v3', scaler=scaler)
X_tgt2, y_tgt2, _    = load_nf_dataset_polars('seyhed/nf-cicids2018-v3', scaler=scaler)

# Helper to create Split Loaders (Rigorous Evaluation)
def create_split_loaders(X, y, batch_size=config.batch_size):
    # 1. Split 70/30 into Train (Adapt) partition and Test partition
    # 70% for Training/Adaptation, 30% for Testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # 2. Adaptation Loader: Use ONLY Benign samples from Train partition
    mask_benign = y_train == 0
    X_adapt = X_train[mask_benign]
    y_adapt = y_train[mask_benign]
    loader_adapt = DataLoader(TensorDataset(torch.tensor(X_adapt), torch.tensor(y_adapt)), batch_size=batch_size, shuffle=True)

    # 3. Test Loader: Use the ENTIRE Test partition (Unseen Benign + Attacks)
    loader_test = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=batch_size, shuffle=False)

    return loader_adapt, loader_test

# Create Loaders

# Source (UNSW)
loader_src_train, loader_src_test = create_split_loaders(X_src, y_src, config.batch_size)

# Target 1 (ToN-IoT)
loader_tgt1_adapt, loader_tgt1_test = create_split_loaders(X_tgt1, y_tgt1, config.batch_size)

# Target 2 (CICIDS)
loader_tgt2_adapt, loader_tgt2_test = create_split_loaders(X_tgt2, y_tgt2, config.batch_size)

print(f"\nData Split Summary (70/30 Split Applied):")
print(f"  Src Train (Benign): {len(loader_src_train.dataset)} | Src Test (Mixed): {len(loader_src_test.dataset)}")
print(f"  Tgt1 Adapt (Benign): {len(loader_tgt1_adapt.dataset)} | Tgt1 Test (Mixed): {len(loader_tgt1_test.dataset)}")
print(f"  Tgt2 Adapt (Benign): {len(loader_tgt2_adapt.dataset)} | Tgt2 Test (Mixed): {len(loader_tgt2_test.dataset)}")


Using cpu. Starting KAN-IDS Experiment...

⚡ Loading seyhed/nf-unsw-nb15-v3 with Polars...


100%|██████████| 112M/112M [00:03<00:00, 31.7MB/s]

Extracting files...


   -> Shape: (100000, 49) | Attack Rate: 5.35%
⚡ Loading seyhed/nf-ton-iot-v3 with Polars...


100%|██████████| 433M/433M [00:11<00:00, 38.7MB/s]

Extracting files...


   -> Shape: (100000, 49) | Attack Rate: 38.97%
⚡ Loading seyhed/nf-cicids2018-v3 with Polars...


100%|██████████| 801M/801M [00:20<00:00, 40.6MB/s]

Extracting files...


   -> Shape: (100000, 49) | Attack Rate: 12.89%

Data Split Summary (70/30 Split Applied):
  Src Train (Benign): 66254 | Src Test (Mixed): 30000
  Tgt1 Adapt (Benign): 42720 | Tgt1 Test (Mixed): 30000
  Tgt2 Adapt (Benign): 60974 | Tgt2 Test (Mixed): 30000


In [ ]:
# 2. INIT MODEL & MEMORY
model = KanAE(input_dim=X_src.shape[1], latent_dim=config.latent_dim)
memory = ReplayBuffer(capacity=config.memory_capacity)

print("\n" + "="*40)
print("PHASE 1: PRE-TRAINING (SOURCE: UNSW - Benign Only)")
print("="*40)
# Train on loader_src_train (Benign)
train_continual(model, loader_src_train, memory, epochs=config.pretrain_epochs, device=device, mode='source')



PHASE 1: PRE-TRAINING (SOURCE: UNSW - Benign Only)
   Epoch 1 Loss: 0.0164
   Epoch 2 Loss: 0.0016
   Epoch 3 Loss: 0.0008
   Epoch 4 Loss: 0.0006
   Epoch 5 Loss: 0.0005
   Epoch 6 Loss: 0.0004
   Epoch 7 Loss: 0.0003
   Epoch 8 Loss: 0.0002
   Epoch 9 Loss: 0.0002
   Epoch 10 Loss: 0.0001
   Epoch 11 Loss: 0.0001
   Epoch 12 Loss: 0.0001
   Epoch 13 Loss: 0.0001
   Epoch 14 Loss: 0.0001
   Epoch 15 Loss: 0.0001


In [ ]:
print("\n" + "="*40)
print("PHASE 2: CROSS-EVALUATION (BASELINE)")
print("="*40)
# Evaluate on TEST loaders (Mixed Benign + Attack)
evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (UNSW)")
evaluate_anomaly_detection(model, loader_tgt1_test, device, desc="Target 1 (ToN-IoT) [Zero-Shot]")
evaluate_anomaly_detection(model, loader_tgt2_test, device, desc="Target 2 (CICIDS) [Zero-Shot]")


PHASE 2: CROSS-EVALUATION (BASELINE)
   [Source (UNSW)] Best F1: 0.9579 | Acc: 99.54% | Thresh: 0.0023
   [Target 1 (ToN-IoT) [Zero-Shot]] Best F1: 0.5610 | Acc: 39.03% | Thresh: 100.2319
   [Target 2 (CICIDS) [Zero-Shot]] Best F1: 0.2531 | Acc: 24.21% | Thresh: 46.7349


(0.2421, np.float64(0.2530797279762392))

In [ ]:
print("\n" + "="*40)
print("PHASE 3: CONTINUAL ADAPTATION (TARGET 1: ToN-IoT)")
print("="*40)
# Adapt to ToN-IoT Benign samples while replaying UNSW Benign
train_continual(model, loader_tgt1_adapt, memory, epochs=config.adapt_epochs, device=device, mode='adapt')



PHASE 3: CONTINUAL ADAPTATION (TARGET 1: ToN-IoT)
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
   Adaptation Epoch 1 Complete. Avg Update Loss: 1.6763


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [ ]:
print("\n[POST-ADAPTATION RESULTS]")
# Evaluate on mixed test sets
evaluate_anomaly_detection(model, loader_tgt1_test, device, desc="Target 1 (ToN-IoT) [Adapted]")
evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (UNSW) [Retention Check]")


[POST-ADAPTATION RESULTS]
   [Target 1 (ToN-IoT) [Adapted]] Best F1: 0.7145 | Acc: 76.81% | Thresh: 0.0084
   [Source (UNSW) [Retention Check]] Best F1: 0.9409 | Acc: 99.34% | Thresh: 0.0023


(0.9934333333333333, np.float64(0.9408586009707806))

In [ ]:
print("\n" + "="*40)
print("PHASE 4: CONTINUAL ADAPTATION (TARGET 2: CICIDS)")
print("="*40)
# Adapt to CICIDS Benign
train_continual(model, loader_tgt2_adapt, memory, epochs=config.adapt_epochs, device=device, mode='adapt')

print("\n[POST-ADAPTATION RESULTS]")
evaluate_anomaly_detection(model, loader_tgt2_test, device, desc="Target 2 (CICIDS) [Adapted]")
evaluate_anomaly_detection(model, loader_src_test, device, desc="Source (UNSW) [Retention Check]")



PHASE 4: CONTINUAL ADAPTATION (TARGET 2: CICIDS)
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
       [Online Adapt] Batch F1: 0.0000
   Adaptation Epoch 1 Complete. Avg Update Loss: 0.0046

[POST-ADAPTATION RESULTS]
   [Target 2 (CICIDS) [Adapted]] Best F1: 0.3855 | Acc: 61.83% | Thresh: 0.0146
   [Source (UNSW) [Retention Check]] Best F1: 0.9323 | Acc: 99.27% | Thresh: 0.0033


(0.9927, np.float64(0.9323029365806058))